In [ ]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl

from dash import dcc
from dash import html
from dash import dash_table

from dash.dependencies import Input
from dash.dependencies import Output

import plotly.express as px
import pandas as pd
import base64

from CRUD_Python_Module_Enhanced import AnimalShelter


# MongoDB username
username = "aacuser"

# Enter your MongoDB password here.
password = "YOUR_PASSWORD_HERE"


# Create database object.
db = AnimalShelter(
    username,
    password
)


# Create the database indexes used
# by the enhanced search functionality.
created_indexes = db.create_indexes()

print(
    "Database indexes:",
    created_indexes
)


# --------------------------------------------------
# Convert MongoDB records to a Pandas DataFrame.
# --------------------------------------------------
def records_to_dataframe(records):

    frame = pd.DataFrame.from_records(
        records
    )

    # MongoDB's _id field is not needed
    # by the dashboard.
    if "_id" in frame.columns:

        frame.drop(
            columns=["_id"],
            inplace=True
        )

    return frame


# Load the original database records.
df = records_to_dataframe(
    db.read({})
)

df.head()

In [ ]:
app = JupyterDash(__name__)


# --------------------------------------------------
# Load the Grazioso Salvare logo.
# --------------------------------------------------
image_filename = "Grazioso Salvare Logo.png"

logo = None

try:

    encoded_image = base64.b64encode(
        open(
            image_filename,
            "rb"
        ).read()
    ).decode()

    logo = html.Img(
        src=(
            "data:image/png;base64,"
            + encoded_image
        ),
        height="100px"
    )

except FileNotFoundError:

    logo = html.Div()


# --------------------------------------------------
# Create options for the sex dropdown.
# --------------------------------------------------
sex_options = []

if (
    not df.empty
    and "sex_upon_outcome"
    in df.columns
):

    values = sorted(
        df[
            "sex_upon_outcome"
        ].dropna().unique()
    )

    for value in values:

        sex_options.append(
            {
                "label": str(value),
                "value": value
            }
        )


# --------------------------------------------------
# Dashboard Layout
# --------------------------------------------------
app.layout = html.Div([

    html.Center([

        logo,

        html.H1(
            "Grazioso Salvare Dashboard"
        ),

        html.H4(
            "Chase Sears - "
            "CS 499 Database Enhancement"
        )

    ]),

    html.Hr(),


    # ==============================================
    # ADVANCED DATABASE SEARCH
    # ==============================================

    html.H3(
        "Advanced Animal Search"
    ),


    html.Div([

        # Breed
        html.Div([

            html.Label(
                "Breed contains"
            ),

            dcc.Input(
                id="breed-filter",
                type="text",
                placeholder="Example: Labrador"
            )

        ]),


        # Minimum Age
        html.Div([

            html.Label(
                "Minimum age (weeks)"
            ),

            dcc.Input(
                id="min-age",
                type="number",
                min=0
            )

        ]),


        # Maximum Age
        html.Div([

            html.Label(
                "Maximum age (weeks)"
            ),

            dcc.Input(
                id="max-age",
                type="number",
                min=0
            )

        ]),


        # Sex
        html.Div([

            html.Label(
                "Sex"
            ),

            dcc.Dropdown(
                id="sex-filter",
                options=sex_options,
                placeholder="Any"
            )

        ]),


        # Location
        html.Div([

            html.Label(
                "Location / outcome subtype"
            ),

            dcc.Input(
                id="location-filter",
                type="text",
                placeholder="Example: Partner"
            )

        ])

    ],

    style={
        "display": "grid",

        "gridTemplateColumns":
            "repeat(5, 1fr)",

        "gap": "10px"
    }),


    html.Br(),


    # Shows how many records matched.
    html.Div(
        id="search-status"
    ),


    html.Hr(),


    # ==============================================
    # DATA TABLE
    # ==============================================

    dash_table.DataTable(

        id="datatable-id",

        columns=[
            {
                "name": column,
                "id": column,
                "selectable": True
            }

            for column in df.columns
        ],

        data=df.to_dict(
            "records"
        ),

        page_size=10,

        sort_action="native",

        filter_action="native",

        row_selectable="single",

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left"
        }

    ),


    html.Hr(),


    # ==============================================
    # DATABASE REPORTING
    # ==============================================

    html.H3(
        "Database Reporting"
    ),


    dcc.Dropdown(

        id="report-group",

        options=[

            {
                "label": "Breed",
                "value": "breed"
            },

            {
                "label": "Sex",
                "value": "sex"
            },

            {
                "label":
                    "Location / Outcome Subtype",

                "value": "location"
            },

            {
                "label": "Outcome Type",
                "value": "outcome"
            }

        ],

        value="breed",

        clearable=False,

        style={
            "maxWidth": "400px"
        }

    ),


    dcc.Graph(
        id="report-chart"
    ),


    html.Hr(),


    # ==============================================
    # ORIGINAL VISUALIZATIONS
    # ==============================================

    html.Div(

        style={
            "display": "flex"
        },

        children=[

            dcc.Graph(

                id="graph-chart",

                style={
                    "width": "50%"
                }

            ),


            html.Div(

                id="map-id",

                style={
                    "width": "50%"
                }

            )

        ]

    )

])

In [ ]:
@app.callback(

    [
        Output(
            "datatable-id",
            "data"
        ),

        Output(
            "search-status",
            "children"
        )
    ],

    [
        Input(
            "breed-filter",
            "value"
        ),

        Input(
            "min-age",
            "value"
        ),

        Input(
            "max-age",
            "value"
        ),

        Input(
            "sex-filter",
            "value"
        ),

        Input(
            "location-filter",
            "value"
        )
    ]

)
def update_dashboard(
        breed,
        min_age,
        max_age,
        sex,
        location):

    # Validate the age range.
    if (
        min_age is not None
        and max_age is not None
        and min_age > max_age
    ):

        return (
            [],
            "Validation error: "
            "minimum age cannot be "
            "greater than maximum age."
        )


    # Build the MongoDB query.
    query = db.build_search_query(

        breed,

        min_age,

        max_age,

        sex,

        location
    )


    # Execute the query.
    records = db.read(
        query
    )


    # Convert the results.
    filtered = records_to_dataframe(
        records
    )


    # Update the dashboard.
    return (

        filtered.to_dict(
            "records"
        ),

        str(len(filtered))
        + " matching animal record(s)."

    )

In [ ]:
@app.callback(

    Output(
        "report-chart",
        "figure"
    ),

    [
        Input(
            "report-group",
            "value"
        ),

        Input(
            "breed-filter",
            "value"
        ),

        Input(
            "min-age",
            "value"
        ),

        Input(
            "max-age",
            "value"
        ),

        Input(
            "sex-filter",
            "value"
        ),

        Input(
            "location-filter",
            "value"
        )
    ]

)
def update_report(
        group_by,
        breed,
        min_age,
        max_age,
        sex,
        location):

    try:

        # Use the same search criteria
        # that are being used by the table.
        query = db.build_search_query(

            breed,

            min_age,

            max_age,

            sex,

            location
        )

    except (TypeError, ValueError):

        return px.bar(
            title="Database Summary Report"
        )


    # Run the MongoDB aggregation.
    report = db.aggregate_report(

        group_by=group_by,

        match=query,

        limit=10
    )


    # Convert the aggregation results.
    report_df = pd.DataFrame(
        report
    )


    if report_df.empty:

        return px.bar(
            title="Database Summary Report"
        )


    report_df.rename(

        columns={
            "_id": "category"
        },

        inplace=True
    )


    # Display the aggregation results.
    figure = px.bar(

        report_df,

        x="category",

        y="count",

        title=(
            "Top "
            + group_by.title()
            + " Groups from "
              "MongoDB Aggregation"
        )

    )

    return figure

In [ ]:
@app.callback(

    Output(
        "graph-chart",
        "figure"
    ),

    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        )
    ]

)
def update_breed_chart(
        view_data):

    dff = pd.DataFrame(
        view_data or []
    )


    if (
        dff.empty
        or "breed" not in dff.columns
    ):

        return px.pie(
            title="Preferred Animals"
        )


    # Find the ten most common breeds.
    top = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(10)
        .reset_index()
    )


    top.columns = [
        "breed",
        "count"
    ]


    figure = px.pie(

        top,

        names="breed",

        values="count",

        title="Preferred Animals",

        hole=0.3
    )


    return figure


In [ ]:
@app.callback(

    Output(
        "datatable-id",
        "style_data_conditional"
    ),

    [
        Input(
            "datatable-id",
            "selected_columns"
        )
    ]

)
def update_styles(
        selected_columns):

    styles = []

    for column in (
        selected_columns or []
    ):

        styles.append({

            "if": {
                "column_id": column
            },

            "background_color":
                "#D2F3FF"

        })

    return styles

In [ ]:
@app.callback(

    Output(
        "map-id",
        "children"
    ),

    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),

        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]

)
def update_map(
        view_data,
        selected_rows):

    if not view_data:

        return html.P(
            "No animal record is "
            "available to map."
        )


    dff = pd.DataFrame(
        view_data
    )


    # Use the selected row.
    # If no row has been selected,
    # use the first result.
    if selected_rows:

        row = selected_rows[0]

    else:

        row = 0


    # Fields required by the map.
    required_fields = {

        "location_lat",

        "location_long",

        "breed",

        "name"
    }


    if not required_fields.issubset(
        dff.columns
    ):

        return html.P(
            "Location data is not "
            "available for the "
            "selected record."
        )


    try:

        latitude = float(
            dff.iloc[row][
                "location_lat"
            ]
        )


        longitude = float(
            dff.iloc[row][
                "location_long"
            ]
        )


    except (TypeError, ValueError):

        return html.P(
            "The selected record "
            "does not contain "
            "valid coordinates."
        )


    # Build the map.
    return dl.Map(

        style={
            "width": "100%",
            "height": "500px"
        },

        center=[
            latitude,
            longitude
        ],

        zoom=10,

        children=[

            dl.TileLayer(),

            dl.Marker(

                position=[
                    latitude,
                    longitude
                ],

                children=[

                    dl.Tooltip(

                        str(
                            dff.iloc[row][
                                "breed"
                            ]
                        )

                    ),

                    dl.Popup([

                        html.H4(
                            "Animal Name"
                        ),

                        html.P(

                            str(
                                dff.iloc[row][
                                    "name"
                                ]
                            )

                        )

                    ])

                ]

            )

        ]

    )
app.run_server(
    mode="jupyterlab"
)